In [19]:
!pip uninstall -y torchao

In [20]:
# !pip install -q torchao>=0.16.0 peft transformers accelerate

In [43]:
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, AutoModelForMultipleChoice, Trainer,TrainingArguments,DataCollatorForMultipleChoice
from peft import LoraConfig, get_peft_model, TaskType
import pandas as pd
import torch
import torch.nn.functional as F
from datasets import Dataset

In [24]:
train_data = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')

# Multiple-Choice Data Formatting
In this section, you will convert the Kaggle MCQ format into the structure required by multiple-choice models. Each question has one prompt and five options and each option must be paired with the prompt separately.

Q1. Label Encoding
Convert the answer column in train.csv into numeric labels using the following mapping:
A = 0
B = 1
C = 2
D = 3
E = 4

What is the encoded numeric label for the row at index 150?

In [25]:
# Label encoding
label_map = {
    "A": 0,
    "B": 1,
    "C": 2,
    "D": 3,
    "E": 4
}

train_data["answer_label"] = train_data["answer"].map(label_map)

# Encoded label at index 150
print(train_data.loc[150, "answer_label"])

# To check the original answer and their encoding matches as expected
print("Original:", train_data.loc[150, "answer"])
print("Encoded :", train_data.loc[150, "answer_label"])

2
Original: C
Encoded : 2


Q2. Prompt-Option Formatting
For row index 0, create the Option B input using exactly this format:
str(prompt) + " [SEP] " + str(option_B)

What is the exact character length of this formatted input string?

In [26]:
formatted_input = str(train_data.loc[0,"prompt"]) + " [SEP] " + str(train_data.loc[0,"B"])

print(f"Formatted input :\n{formatted_input}")
print(f"\nCharacter length : {len(formatted_input)}")

Formatted input :
Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options. [SEP] Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the past is a present awareness of having been, and the relationship to the future involves anticipating a potential possibility, task, or engagement.

Character length : 407


# Tokenization for Multiple-Choice Models
Multiple-choice models expect inputs in the shape:
batch_size x num_choices x sequence_length

Since each question has five options, every row becomes five tokenized sequences.

Q3. Single-Row MCQ Tokenization
Using bert-base-uncased, tokenize the five formatted inputs for row index 0 with:
padding = "max_length"
truncation = True
max_length = 128
return_tensors = "pt"

After reshaping for a multiple-choice model, the final input_ids tensor has shape:
[1, 5, 128]

What is the value of the second dimension?

In [27]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Select row 0
row = train_data.iloc[0]

# Create the 5 prompt-option pairs
choices = [
    str(row["prompt"]) + " [SEP] " + str(row["A"]),
    str(row["prompt"]) + " [SEP] " + str(row["B"]),
    str(row["prompt"]) + " [SEP] " + str(row["C"]),
    str(row["prompt"]) + " [SEP] " + str(row["D"]),
    str(row["prompt"]) + " [SEP] " + str(row["E"]),
]

# Tokenize
encoding = tokenizer(
    choices,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

print("Before reshape:")
print(encoding["input_ids"].shape)

# Add batch dimension for MultipleChoice models
encoding = {k: v.unsqueeze(0) for k, v in encoding.items()}

print("\nAfter reshape:")
print("input_ids:", encoding["input_ids"].shape)
print("attention_mask:", encoding["attention_mask"].shape)

# If token_type_ids exist
if "token_type_ids" in encoding:
    print("token_type_ids:", encoding["token_type_ids"].shape)

print(f"Value of the second dimension : {encoding["input_ids"].shape[1]}")

Before reshape:
torch.Size([5, 128])

After reshape:
input_ids: torch.Size([1, 5, 128])
attention_mask: torch.Size([1, 5, 128])
token_type_ids: torch.Size([1, 5, 128])
Value of the second dimension : 5


Q4. Batch MCQ Tokenization
Tokenize the first 16 rows of train.csv as multiple-choice examples.
Each row has 5 choices.
Each choice is tokenized to length 128.

The final input_ids tensor has shape:
[16, 5, 128]

How many total token positions are in this tensor?

In [28]:
batch_size = 16
num_choices = 5
max_length = 128

total_token_positions = batch_size * num_choices * max_length
print(f"Total token positions are in this tensor : {total_token_positions}")

Total token positions are in this tensor : 10240


# Multiple-Choice Model Outputs
AutoModelForMultipleChoice produces one logit score for each answer option. For this competition, the model outputs five logits corresponding to A, B, C, D, and E.

Q5. Multiple-Choice Logits
Load bert-base-uncased using AutoModelForMultipleChoice.
Tokenize row index 0 as 5 choices and pass it through the model.

The output logits tensor has shape:
[1, 5]

How many logits are produced for one question?

In [29]:
# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModelForMultipleChoice.from_pretrained("bert-base-uncased")

# Select first question
row = train_data.iloc[0]

# Create the 5 prompt-option pairs
choices = [
    f"{row['prompt']} [SEP] {row['A']}",
    f"{row['prompt']} [SEP] {row['B']}",
    f"{row['prompt']} [SEP] {row['C']}",
    f"{row['prompt']} [SEP] {row['D']}",
    f"{row['prompt']} [SEP] {row['E']}",
]

# Tokenize
encoding = tokenizer(
    choices,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

# Add batch dimension
encoding = {k: v.unsqueeze(0) for k, v in encoding.items()}

# Forward pass
with torch.no_grad():
    outputs = model(**encoding)

print(outputs.logits.shape)
print(outputs.logits)
print("Number of logits:", outputs.logits.shape[1])

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


torch.Size([1, 5])
tensor([[-0.9734, -0.9547, -0.9786, -0.9728, -1.0132]])
Number of logits: 5


Q6. Supervised Loss Tensor
For row index 0, pass the tokenized 5-choice input into AutoModelForMultipleChoice along with the correct encoded label.

The model returns a scalar loss tensor.

How many dimensions does this loss tensor have?

In [30]:
# Encode labels
label_map = {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4}
train_data["label"] = train_data["answer"].map(label_map)

# Select row 0
row = train_data.iloc[0]

# Create the 5 prompt-option pairs
choices = [
    f"{row['prompt']} [SEP] {row['A']}",
    f"{row['prompt']} [SEP] {row['B']}",
    f"{row['prompt']} [SEP] {row['C']}",
    f"{row['prompt']} [SEP] {row['D']}",
    f"{row['prompt']} [SEP] {row['E']}",
]

# Tokenize
encoding = tokenizer(
    choices,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

# Add batch dimension
encoding = {k: v.unsqueeze(0) for k, v in encoding.items()}

# Correct label
labels = torch.tensor([row["label"]])

# Forward pass with labels
outputs = model(**encoding, labels=labels)

print("Loss:", outputs.loss)
print("Loss shape:", outputs.loss.shape)
print("Loss dimensions:", outputs.loss.ndim)

Loss: tensor(1.5857, grad_fn=<NllLossBackward0>)
Loss shape: torch.Size([])
Loss dimensions: 0


# LoRA for Efficient Fine-Tuning
LoRA freezes most of the original model and trains only a small number of adapter parameters. This makes fine-tuning faster and more memory-efficient.

Q7. LoRA Trainable Parameters
Apply LoRA to the bert-base-uncased multiple-choice model using:
r = 8
lora_alpha = 16
target_modules = ["query", "value"]
lora_dropout = 0.1
bias = "none"
task_type = TaskType.SEQ_CLS

Count trainable parameters using:
sum(p.numel() for p in model.parameters() if p.requires_grad)

How many parameters are trainable?

In [31]:
# LoRA configuration
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS,
)

# Apply LoRA
model = get_peft_model(model, lora_config)

# Count trainable parameters
trainable_params = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)

print("Trainable parameters:", trainable_params)

# Optional: detailed summary
model.print_trainable_parameters()

Trainable parameters: 295681
trainable params: 295,681 || all params: 109,778,690 || trainable%: 0.2693


# Preparing Data for Hugging Face Trainer
Before training, the dataset must be converted into a format that the Hugging Face Trainer can understand: tokenized input_ids, attention_mask, and numeric labels.

Q8. Hugging Face Dataset Preparation
Create a Hugging Face Dataset from the first 100 rows of train.csv.

For each row, create:
input_ids with shape [5, 128]
attention_mask with shape [5, 128]
labels as the encoded answer label

For the first dataset item, input_ids has shape:
[5, 128]

How many tokenized choices are stored in input_ids?

In [34]:
def preprocess(example):
    choices = [
        f"{example['prompt']} [SEP] {example['A']}",
        f"{example['prompt']} [SEP] {example['B']}",
        f"{example['prompt']} [SEP] {example['C']}",
        f"{example['prompt']} [SEP] {example['D']}",
        f"{example['prompt']} [SEP] {example['E']}",
    ]

    encoding = tokenizer(
        choices,
        padding="max_length",
        truncation=True,
        max_length=128,
    )

    return {
        "input_ids": encoding["input_ids"],
        "attention_mask": encoding["attention_mask"],
        "labels": example["answer_label"],
    }

# Create HF Dataset
dataset = Dataset.from_pandas(train_data)
dataset = dataset.map(preprocess)

# Check first item
print("input_ids shape:", (len(dataset[0]["input_ids"]), len(dataset[0]["input_ids"][0])))
print("Number of choices:", len(dataset[0]["input_ids"]))
print("Label:", dataset[0]["answer_label"])

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

input_ids shape: (5, 128)
Number of choices: 5
Label: 1


# Tiny Fine-Tuning and Inference
In this section, you will run a very small LoRA fine-tuning job using Hugging Face Trainer. Then you will use the fine-tuned model to produce probabilities for the answer options.

Q9. Tiny LoRA Fine-Tuning
Fine-tune a LoRA multiple-choice model on the first 32 rows using Hugging Face Trainer.

Use the following settings:
max_length = 64
per_device_train_batch_size = 4
gradient_accumulation_steps = 1
max_steps = 4

What is the final global_step reported by the Trainer?

In [40]:
# Encode labels
label_map = {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4}
train_data["labels"] = train_data["answer"].map(label_map)

# ----------------------------
# Hugging Face Dataset
# ----------------------------
dataset = Dataset.from_pandas(train_data)

# ----------------------------
# Tokenizer
# ----------------------------
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# ----------------------------
# Tokenization
# ----------------------------
def preprocess(example):

    choices = [
        f"{example['prompt']} [SEP] {example['A']}",
        f"{example['prompt']} [SEP] {example['B']}",
        f"{example['prompt']} [SEP] {example['C']}",
        f"{example['prompt']} [SEP] {example['D']}",
        f"{example['prompt']} [SEP] {example['E']}",
    ]

    tokenized = tokenizer(
        choices,
        padding="max_length",
        truncation=True,
        max_length=64,
    )

    tokenized["labels"] = example["labels"]

    return tokenized

dataset = dataset.map(preprocess)

# Keep only required columns
dataset = dataset.remove_columns(
    [c for c in dataset.column_names
     if c not in ["input_ids", "attention_mask", "labels"]]
)

dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

# ----------------------------
# Model
# ----------------------------
model = AutoModelForMultipleChoice.from_pretrained(
    "bert-base-uncased"
)

# ----------------------------
# LoRA
# ----------------------------
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS,
)

model = get_peft_model(model, lora_config)

# Optional
model.print_trainable_parameters()

# ----------------------------
# Data Collator
# ----------------------------
data_collator = DataCollatorForMultipleChoice(tokenizer)

# ----------------------------
# Training Arguments
# ----------------------------
training_args = TrainingArguments(
    output_dir="./mcq_lora",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    max_steps=4,
    logging_steps=1,
    save_strategy="no",
    report_to="none",
)

# ----------------------------
# Trainer
# ----------------------------
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    data_collator=data_collator,
)

# ----------------------------
# Train
# ----------------------------
train_result = trainer.train()

print("\nFinal global_step:", train_result.global_step)
print("Trainer global_step:", trainer.state.global_step)

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 295,681 || all params: 109,778,690 || trainable%: 0.2693


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
1,1.692253
2,1.715489
3,1.639730
4,1.628672



Final global_step: 4
Trainer global_step: 4


Q10. Probability Assigned to Option E After Fine-Tuning
Using the fine-tuned LoRA model from Q9, run inference on row index 0 and apply softmax to the logits.

What is the probability assigned to Option E?

Round your answer to 4 decimal places.

In [44]:
model.eval()

row = train_data.iloc[0]

choices = [
    f"{row['prompt']} [SEP] {row['A']}",
    f"{row['prompt']} [SEP] {row['B']}",
    f"{row['prompt']} [SEP] {row['C']}",
    f"{row['prompt']} [SEP] {row['D']}",
    f"{row['prompt']} [SEP] {row['E']}",
]

encoding = tokenizer(
    choices,
    padding="max_length",
    truncation=True,
    max_length=64,
    return_tensors="pt",
)

# Add batch dimension
encoding = {k: v.unsqueeze(0) for k, v in encoding.items()}

with torch.no_grad():
    outputs = model(**encoding)

logits = outputs.logits
probs = F.softmax(logits, dim=1)

print("Logits:")
print(logits)

print("\nProbabilities:")
print(probs)

print("\nProbability of Option E:")
print(round(probs[0, 4].item(), 4))

Logits:
tensor([[0.0354, 0.0565, 0.0198, 0.0452, 0.0502]])

Probabilities:
tensor([[0.1988, 0.2030, 0.1957, 0.2007, 0.2018]])

Probability of Option E:
0.2018
